# **ECI 2026 : Introduction to Decentralized Finance** 
### *By Julien Prat, Louis Latournerie & Bastien Levy-Guinot*

## Goal
The goal of this homework is to learn how to interact with an Ethereum node and retrieve data from the blockchain. To do this, we will use Ethereum's JSON-RPC API: the standard interface exposed by Ethereum nodes (Geth, Erigon, etc.) for querying blockchain data, via **[Alchemy](https://docs.alchemy.com/)**, a node provider that lets us query the blockchain without running our own node.

Cells marked **`# TODO`** are for you to complete. The **[web3.py documentation](https://web3py.readthedocs.io/en/stable/)** lists all available methods along with usage examples and good practices, and should be your go-to reference throughout this homework.

## Prerequisites
1. Create a free API key on **[Alchemy](https://www.alchemy.com)**.
2. Save it in a `.env` file at the root of the project:
   ```
   ALCHEMY_API_KEY=your-actual-alchemy-api-key
   ```

## Structure
- **Part A** — Wallet-Level Data (balance, transactions)
- **Part B** — Interacting with a Smart Contract
- **Part C** — Aave V3 `Borrow` Events (a DeFi protocol event)
- **Part D** — Investigate a real address's activity

---

## Imports

In [ ]:
!pip install requests pandas web3 python-dotenv -q

In [ ]:
import os
from web3 import Web3
import pandas as pd
import dotenv
import requests
from datetime import datetime, timezone

In [ ]:
if "ALCHEMY_API_KEY" not in globals():
    from dotenv import load_dotenv
    load_dotenv()
    ALCHEMY_API_KEY = os.getenv("ALCHEMY_API_KEY")

ALCHEMY_URL = f"https://eth-mainnet.g.alchemy.com/v2/{ALCHEMY_API_KEY}"
print("Alchemy API key loaded.")

In [ ]:
# Create a web3 instance connected to an Ethereum node via Alchemy's HTTP endpoint
w3 = Web3(Web3.HTTPProvider(ALCHEMY_URL))

# Query the node for the number of the latest mined block (sanity check that the connection works)
alchemy_latest_block = w3.eth.get_block_number()
print("Alchemy says latest block is:", alchemy_latest_block)

## A - Collecting Basic Address-Level Data

In [ ]:
WALLET_ADDRESS = "0x0373B4553c9879510Da2a320126900576ef5Bc3C"
BLOCK_NUMBER = 19_974_916

In [ ]:
# TODO: Fetch the ETH balance of WALLET_ADDRESS at block BLOCK_NUMBER.
#
# Remember to call the function at the right block (it defaults to "latest" otherwise),
# and to rescale the result from wei to ETH.
# Check the web3.py documentation (https://web3py.readthedocs.io/en/stable/) to find the
# right method.

balance_eth_alchemy = None  # TODO: fetch the balance at BLOCK_NUMBER and rescale to ETH

print(f"The balance of address {WALLET_ADDRESS} at block {BLOCK_NUMBER} is {balance_eth_alchemy} ETH.")

In [ ]:
# TODO: Fetch the number of transactions sent by WALLET_ADDRESS up to and including BLOCK_NUMBER.
#
# Remember to call the function at the right block (it defaults to "latest" otherwise).
# Check the web3.py documentation (https://web3py.readthedocs.io/en/stable/) to find the
# right method.

tx_count = None  # TODO: fetch the transaction count at BLOCK_NUMBER

print(f"The address {WALLET_ADDRESS} has sent {tx_count} transactions until block {BLOCK_NUMBER}.")

## B - Interacting with Smart Contracts

### ERC-20 Standard (See the [Ethereum documentation](https://ethereum.org/developers/docs/standards/tokens/erc-20/))

The most widely used Ethereum token standards are:
- **ERC-20**: fungible tokens (interchangeable units, like currencies)
- **ERC-721**: non-fungible tokens (NFTs, each token is unique)
- **ERC-1155**: multi-token standard (fungible and non-fungible tokens in a single contract, common in gaming)

**What ERC-20 implements.** ERC-20 is the standard for *fungible* tokens — units that are identical and interchangeable, just like currency units. It is used for stablecoins such as **USDC** and **USDT** (pegged to the US dollar), for wrapped assets such as **WBTC** (Bitcoin represented on Ethereum), and for governance/utility tokens issued by countless projects.

**The ERC-20 interface.** Concretely, ERC-20 is a specification that a smart contract must implement, consisting of mandatory functions and events:

- Functions: `totalSupply()`, `balanceOf(address)`, `transfer(address, uint256)`, `approve(address, uint256)`, `allowance(address, address)`, `transferFrom(address, address, uint256)`
- Events: `Transfer(address, address, uint256)`, `Approval(address, address, uint256)`

Any contract exposing this exact interface is an ERC-20 token — there is no separate "token" object on Ethereum. What we casually call a "token" is simply a smart contract deployed on-chain that implements the ERC-20 standard.

**Why standardization matters.** Ethereum is a general-purpose platform: anyone can deploy a smart contract that represents an asset (a currency, a share, a collectible, ...). Without a common interface, every wallet, exchange, and dApp would need custom code to interact with every single contract. Standards solve this by defining a common set of functions and events that contracts must implement, so that any compliant tool can interact with any compliant token out of the box.

**⚠️ ETH is not a token.** ETH is the native cryptocurrency of the Ethereum blockchain. It is not a smart contract and does not follow the ERC-20 standard. It's used to pay gas fees and is tracked directly at the protocol level (via account balances), not through a `balanceOf` call. Do not confuse it with **WETH** (Wrapped ETH), which is an ERC-20 contract that lets ETH be represented and used as if it were a standard token (e.g. to interact with DeFi protocols that expect ERC-20 tokens).

### What is an ABI? (See the [Ethereum documentation](https://ethereum.org/developers/docs/smart-contracts/interacting/))

A deployed smart contract only exists on-chain as raw compiled bytecode. The ABI (Application Binary Interface) is a JSON description of a contract's functions and events (names, argument types, return types) that lets a library like web3.py translate a readable call such as contract.functions.balanceOf(address).call() into the correct low-level calldata, and decode the raw response back into a usable value. Without it, you would have to compute function selectors and encode/decode arguments by hand from the raw bytecode. The ABI is not stored in the contract itself. It's published separately (e.g. on Etherscan, or by the protocol). Since all ERC-20 contracts share the same interface, one generic ERC-20 ABI works for any ERC-20 token (USDC, USDT, WBTC, ...).

In [ ]:
# TODO: Retrieve the address and ABI of the Wrapped Bitcoin (WBTC) contract from Etherscan,
# then use them to fetch the WBTC balance of WALLET_ADDRESS at block BLOCK_NUMBER.
#
# 1. WBTC_ABI: the contract's ABI (WBTC_ADDRESS is already given below).
# 2. wbtc_contract: a web3 contract instance built from WBTC_ADDRESS and WBTC_ABI.
# 3. balance: the result of calling balanceOf(WALLET_ADDRESS) at block BLOCK_NUMBER,
#    rescaled using the contract's number of decimals.

WBTC_ADDRESS = "0x2260FAC5E5542a773Aa44fBCfeDf7C193bc2C599"
WBTC_ABI = None  # TODO: fetch from Etherscan

wbtc_contract = None  # TODO: build the web3 contract instance
balance = None  # TODO: query balanceOf and rescale by the number of decimals

print(f"The balance of address {WALLET_ADDRESS}\nin Wrapped Bitcoin ({WBTC_ADDRESS})\nat block {BLOCK_NUMBER} is {balance}.")

## C - Collecting Aave-v3 Borrow Events

### What is an Event?

An **event** is a piece of data that a smart contract explicitly emits (logs) when something happens during a transaction, e.g. a transfer, a deposit, a new borrow. Events are stored in the transaction's **logs**, separately from the contract's state, which makes them cheap to write and, crucially, easy to query and filter afterwards.

The ERC-20 standard requires every token contract to emit a **`Transfer(from, to, value)`** event on every transfer. Since `from` and `to` are **indexed** parameters, a node can filter logs directly by these fields via the `eth_getLogs` method. This means you can retrieve **every transfer of a given token, or every transfer involving a given address, in one query**, without scanning the blockchain transaction by transaction and checking whether each one calls `transfer()`. Events are exactly what makes this kind of large-scale, address-level or token-level data collection practical. You can learn more on the [Ethereum documentation](https://ethereum.org/developers/tutorials/logging-events-smart-contracts/).

### Structure of an Event Log

Events are retrieved by querying `eth_getLogs` with a filter: a contract `address`, a block range (`fromBlock`/`toBlock`), and optionally a list of `topics` to filter on. The node then returns every matching log, each with the following structure:

- **`topics[0]`** The **event signature**. This is the `keccak256` hash of the event's name and argument types, e.g. `keccak256("Transfer(address,address,uint256)")`. This is what tells you which event was emitted, and it's the first thing you filter on when querying (e.g. "give me all logs where `topics[0]` matches the `Transfer` signature").
- **`topics[1]`, `topics[2]`, `topics[3]`** The event's **indexed** parameters, in order. For `Transfer(address indexed from, address indexed to, uint256 value)`, `topics[1]` is `from` and `topics[2]` is `to`. Because they live in `topics`, these fields can be filtered directly in the query itself, e.g. "give me all `Transfer` logs where `to` is my address", without downloading and scanning irrelevant logs.
- **`data`** A single ABI-encoded blob holding all **non-indexed** parameters (here, `value`, the amount transferred). This part is not filterable at query time: it's just returned as raw bytes and must be decoded afterwards using the event's ABI.

So concretely, querying "all Transfers to my address" means calling `eth_getLogs` with the `Transfer` signature as `topics[0]` and your address as `topics[2]`, the node does the filtering for you, and you only need to decode `data` to get the transferred amount.

With `web3.py`, this filtering is done through `w3.eth.filter`. For example, to get every `Transfer` event emitted by the USDC contract over a range of blocks:

```python
events_filter = w3.eth.filter(
    {
        "address": USDC_ADDRESS,
        "fromBlock": FROM_BLOCK,
        "toBlock": TO_BLOCK,
        "topics": [
            "0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef"
        ],
    }
)
transfers_list = events_filter.get_all_entries()
```

Here, `address` restricts the search to the USDC contract, `fromBlock`/`toBlock` set the block range, and `topics[0]` is the precomputed `keccak256("Transfer(address,address,uint256)")` signature, since we only filter on `topics[0]`, this returns every transfer of USDC in that range, regardless of sender or recipient. To narrow it down to transfers involving a specific address, you'd add it (left-padded to 32 bytes) as `topics[1]` (for `from`) or `topics[2]` (for `to`).

### Aave-v3 Borrow Events

**Aave V3** is one of the largest decentralized lending protocols on Ethereum: users deposit assets as collateral and can borrow other assets against it, all through a single smart contract called the **Pool**. Every time a user borrows, the Pool contract emits a **`Borrow`** event recording the asset, the amount, and the borrower's address. We will track **`Borrow`** events emitted by the **Aave V3 Pool** contract on Ethereum mainnet:
[`0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2`](https://etherscan.io/address/0x87870bca3f3fd6335c3f4ce8392d69350b4fa4e2).

In [ ]:
# TODO: Collect all `Borrow` events emitted by the Aave V3 Pool contract at block 19974916.
#
# 1. Compute BORROW_TOPIC0: the Keccak-256 hash of the `Borrow` event signature
#    (same principle as the Transfer event topic0) you can also retrieve it directly from Etherscan.
# 2. Build borrow_events_filter: an eth_getLogs filter on AAVE_V3_POOL, restricted
#    to [start_block, to_block], with BORROW_TOPIC0 as topics[0].
# 3. Store the resulting logs in borrow_events.

AAVE_V3_POOL = "0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2"
start_block = BLOCK_NUMBER
to_block = BLOCK_NUMBER

BORROW_TOPIC0 = None  # TODO: keccak-256 hash of the Borrow event signature

borrow_events_filter = None  # TODO: eth_getLogs filter (address, block range, topics)

borrow_events = None  # TODO: fetch the logs matching the filter above

In [ ]:
def decode_borrow_event(raw_event) -> dict:
    """Turn one raw Borrow event log into a clean dict."""

    # TODO: decode `amount` from `data`. For a Borrow event, `data` holds FOUR
    # 32-byte words in this order: [user, amount, interestRateMode, borrowRate].
    # Strip the "0x" prefix (if present), then take the SECOND word hex
    # characters 64:128 and convert from hex to int.

    amount = None  # log["data"] -> int

    # TODO: decode `reserve` (the borrowed token) from topics[1].
    # An address is stored right-padded to 32 bytes (64 hex chars); the actual
    # 20-byte address is the LAST 40 hex characters. Don't forget the "0x" prefix.

    reserve = None  # topics[1] -> address, e.g. "0x" + topics[1][-40:]

    # TODO: decode `onBehalfOf` the same way, from topics[2].

    on_behalf_of = None

    return {
        "reserve": reserve,
        "on_behalf_of": on_behalf_of,
        "amount_raw": amount,
        "block": int(raw_event["blockNumber"], 16) if isinstance(raw_event["blockNumber"], str) else raw_event["blockNumber"],
        "tx_hash": raw_event["transactionHash"],
    }


# Decode and print every Borrow event we fetched
for idx, borrow in enumerate(borrow_events):
    print(idx, " : ", decode_borrow_event(raw_event=borrow))

## D - Investigate: what happened to the borrowed funds?

In Part C, the `Borrow` event you decoded had `onBehalfOf` =
[`0x0373b4553c9879510da2a320126900576ef5bc3c`](https://etherscan.io/address/0x0373b4553c9879510da2a320126900576ef5bc3c). This is the same address as `WALLET_ADDRESS` from Part A. Now trace its broader activity to see what it actually *did* with its funds.

You'll fetch **every transfer** involving this address over a much wider window than before and try to reconstruct where the money went.

### D1 - Fetch all transfers in the window

Fetch **every** ERC-20 transfer for `WALLET_ADDRESS` between block `19974916` and block `19975014`, using either `w3.eth.filter` or [Alchemy's Enhanced Transfers API](https://www.alchemy.com/docs/reference/transfers-api-quickstart).

> **⚠️ Alchemy free-tier limit on `eth_getLogs`**
>
> On the free tier, Alchemy restricts `eth_getLogs` requests to a **10-block range**. Querying a wider range will raise an error such as:
>
> ```
> {'code': -32600, 'message': 'Under the Free tier plan, you can make eth_getLogs requests
> with up to a 10 block range...'}
> ```
>
> If you use `w3.eth.filter`, you therefore need to split the range into batches of at most 10 blocks and loop over them. Alternatively, the Enhanced Transfers API is not subject to this restriction and lets you query on a wider range of blocks, and you can include other types of transfers (in particular internal and external) that are not included in a standard event filter. 

In [ ]:
D_START_BLOCK = BLOCK_NUMBER
D_END_BLOCK = 19975014

# TODO: Fetch ALL ERC-20 token transfers for WALLET_ADDRESS between D_START_BLOCK and
# D_END_BLOCK, using either an event log filter (w3.eth.filter) or Alchemy's Enhanced
# Transfers API.
#
# Since a Transfer event indexes "from" and "to" separately, you'll need two queries -
# one filtering on WALLET_ADDRESS as sender, one as recipient - then combine the results.
#
# Build a DataFrame df_transfers with at least the following columns:
# ["hash", "from", "to", "tokenSymbol", "value", "block", "datetime"]
# "value" should be rescaled into a human-readable amount (not raw token units).

df_transfers = None

### D2 - Analyze Activity

Based on your investigation above, write a short paragraph (4-6 sentences):

What were the actions triggered by `0x0373b4553c9879510da2a320126900576ef5bc3c` ?

Do not hesitate to use Etherscan to label the addresses in your DataFrame.

_Your answer here:_

## Bonus Question

Collect all `Borrow` events that occurred on May 29th, 2025. For each one, use [Alchemy's Enhanced Transfers API](https://www.alchemy.com/docs/reference/transfers-api-quickstart) to retrieve all external, internal, and ERC-20 transfers made by the borrower within the 150 blocks following the borrow. Unlike the event-log filtering we used so far, which only captures ERC-20 `Transfer` events, this API also surfaces plain ETH transfers (external) and transfers made internally by smart contracts (internal), giving a much more complete picture of what happened to the borrowed funds. Based on these transfers, can you identify which DeFi protocols the borrowed funds were sent to (AMMs, staking protocols, other lending protocols, bridges, ...)?